In [9]:
%pip install shapely

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 16.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from pathlib import Path
import time
import json

from pathlib import Path
from shapely.geometry import shape, box, mapping
from shapely.ops import unary_union

Path("data").mkdir(exist_ok=True)


In [2]:
# Sample locations across the San Diego region
latitudes = [32.625, 32.875, 33.125, 33.375]
longitudes = [-117.375, -117.125, -116.875, -116.625]

locations = [
    (lat, lon)
    for lat in latitudes
    for lon in longitudes
]

print("Number of locations:", len(locations))
print(locations)


Number of locations: 16
[(32.625, -117.375), (32.625, -117.125), (32.625, -116.875), (32.625, -116.625), (32.875, -117.375), (32.875, -117.125), (32.875, -116.875), (32.875, -116.625), (33.125, -117.375), (33.125, -117.125), (33.125, -116.875), (33.125, -116.625), (33.375, -117.375), (33.375, -117.125), (33.375, -116.875), (33.375, -116.625)]


In [3]:
def download_temperature(lat, lon):
    url = "https://power.larc.nasa.gov/api/projection/daily/point"

    params = {
        "parameters": "T2M_MAX",
        "community": "SB",
        "longitude": lon,
        "latitude": lat,
        "start": "20150101",
        "end": "21001231",
        "format": "CSV",
        "model": "gfdl_esm4",
        "time-standard": "UTC"
    }

    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    # Find actual table header after NASA metadata
    lines = response.text.splitlines()

    header_index = next(
        i for i, line in enumerate(lines)
        if line.strip().startswith("YEAR,MO,DY")
    )

    data = pd.read_csv(
        StringIO("\n".join(lines[header_index:]))
    )

    data["latitude"] = lat
    data["longitude"] = lon

    return data


In [4]:
test_data = download_temperature(32.625, -117.125)

print("Shape:", test_data.shape)
display(test_data.head())


Shape: (31411, 8)


,YEAR,MO,DY,T2M_MAX_SSP126,T2M_MAX_SSP245,T2M_MAX_SSP370,latitude,longitude
0,2015,1,1,15.03,15.17,15.19,32.625,-117.125
1,2015,1,2,16.02,15.94,15.85,32.625,-117.125
2,2015,1,3,16.34,16.23,16.27,32.625,-117.125
3,2015,1,4,15.73,15.66,15.72,32.625,-117.125
4,2015,1,5,16.50,16.75,16.59,32.625,-117.125


In [5]:
all_data = []

for i, (lat, lon) in enumerate(locations):
    print(f"Downloading {i+1}/{len(locations)}: {lat}, {lon}")

    if lat == 32.625 and lon == -117.125:
        data = test_data.copy()
    else:
        data = download_temperature(lat, lon)

    all_data.append(data)
    time.sleep(0.5)

regional_df = pd.concat(all_data, ignore_index=True)

print("Total rows:", len(regional_df))
display(regional_df.head())


Total rows: 502576


,YEAR,MO,DY,T2M_MAX_SSP126,T2M_MAX_SSP245,T2M_MAX_SSP370,latitude,longitude
0,2015,1,1,15.71,15.80,15.83,32.625,-117.375
1,2015,1,2,16.42,16.36,16.29,32.625,-117.375
2,2015,1,3,16.87,16.78,16.80,32.625,-117.375
3,2015,1,4,16.32,16.27,16.33,32.625,-117.375
4,2015,1,5,16.91,17.05,17.00,32.625,-117.375


In [6]:
temp_cols = [
    "T2M_MAX_SSP126",
    "T2M_MAX_SSP245",
    "T2M_MAX_SSP370"
]

# Replace NASA missing values
regional_df[temp_cols] = (
    regional_df[temp_cols].replace(-999, np.nan)
)

# Convert to Fahrenheit
for scenario in ["SSP126", "SSP245", "SSP370"]:
    regional_df[f"MAX_F_{scenario}"] = (
        regional_df[f"T2M_MAX_{scenario}"] * 1.8 + 32
    )

# Create date column
regional_df["date"] = pd.to_datetime(
    regional_df[["YEAR", "MO", "DY"]].rename(
        columns={
            "YEAR": "year",
            "MO": "month",
            "DY": "day"
        }
    )
)

# Save full cleaned regional data
regional_df.to_csv(
    "data/regional_temperatures.csv",
    index=False
)


In [7]:
# Average daily maximum temperature by year,
# month, and geographic location
monthly_df = (
    regional_df
    .groupby(["latitude", "longitude", "YEAR", "MO"])[
        ["MAX_F_SSP126", "MAX_F_SSP245", "MAX_F_SSP370"]
    ]
    .mean()
    .reset_index()
)

monthly_df.to_csv(
    "data/regional_monthly.csv",
    index=False
)

display(monthly_df.head())


,latitude,longitude,YEAR,MO,MAX_F_SSP126,MAX_F_SSP245,MAX_F_SSP370
0,32.625,-117.375,2015,1,69.668194,71.554710,65.665806
1,32.625,-117.375,2015,2,64.862857,70.907000,66.907786
2,32.625,-117.375,2015,3,67.895484,72.484903,66.189548
3,32.625,-117.375,2015,4,71.303000,71.214800,69.734000
4,32.625,-117.375,2015,5,68.168968,69.978839,75.722000


In [11]:
# Load the monthly temperature data
monthly_df = pd.read_csv("data/regional_monthly.csv")

# Load the San Diego County boundary
with open("data/san_diego_county.geojson", "r") as f:
    county_geojson = json.load(f)

print("Temperature data:")
display(monthly_df.head())

print("GeoJSON type:", county_geojson["type"])


Temperature data:


,latitude,longitude,YEAR,MO,MAX_F_SSP126,MAX_F_SSP245,MAX_F_SSP370
0,32.625,-117.375,2015,1,69.668194,71.554710,65.665806
1,32.625,-117.375,2015,2,64.862857,70.907000,66.907786
2,32.625,-117.375,2015,3,67.895484,72.484903,66.189548
3,32.625,-117.375,2015,4,71.303000,71.214800,69.734000
4,32.625,-117.375,2015,5,68.168968,69.978839,75.722000


GeoJSON type: FeatureCollection


In [12]:
# Extract the county boundary geometry
if county_geojson["type"] == "FeatureCollection":
    county_geometries = [
        shape(feature["geometry"])
        for feature in county_geojson["features"]
    ]

elif county_geojson["type"] == "Feature":
    county_geometries = [
        shape(county_geojson["geometry"])
    ]

else:
    county_geometries = [shape(county_geojson)]

county_shape = unary_union(county_geometries)

# Get the unique sample locations
locations_df = monthly_df[
    ["latitude", "longitude"]
].drop_duplicates()

features = []

for _, row in locations_df.iterrows():
    lat = float(row["latitude"])
    lon = float(row["longitude"])

    # Approximate geographic region around sample point
    cell = box(
        lon - 0.125,
        lat - 0.125,
        lon + 0.125,
        lat + 0.125
    )

    # Clip to San Diego County
    clipped_cell = cell.intersection(county_shape)

    if clipped_cell.is_empty:
        continue

    features.append({
        "type": "Feature",
        "properties": {
            "latitude": lat,
            "longitude": lon
        },
        "geometry": mapping(clipped_cell)
    })

temperature_geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open("data/temperature_regions.geojson", "w") as f:
    json.dump(temperature_geojson, f)

print("Temperature regions created:", len(features))
print("Saved temperature_regions.geojson")


Temperature regions created: 16
Saved temperature_regions.geojson


In [13]:
region_shapes = [
    shape(feature["geometry"])
    for feature in features
]

mapped_area = unary_union(region_shapes)

coverage = (
    mapped_area.area / county_shape.area
) * 100

print(f"Approximate county coverage: {coverage:.1f}%")
print("Number of regions:", len(features))


Approximate county coverage: 67.7%
Number of regions: 16
